# Stride — train the floor-plan recognition model (Kaggle)

Use this instead of `colab_train.ipynb` if Colab's paid tiers aren't available in your country (Google publishes a country allowlist for Colab Pro/Pay-As-You-Go — many countries aren't on it). Kaggle Notebooks give **~30 hours/week of free GPU (T4 x2 or P100), no payment method, no country restriction.**

**Before running — two one-time settings, in the panel on the right (⚙️ or "Settings"):**
1. **Accelerator → GPU T4 x2** (or P100 if that's what's offered).
2. **Internet → On** (needed to clone the repo and install packages).

If this is your first time enabling GPU/Internet on Kaggle, it may ask you to verify your phone number first — a one-time Kaggle requirement, unrelated to Stride.

Then: **Run All** (top menu, or ▶▶ in the toolbar). This clones the repo, generates the synthetic dataset, trains the U‑Net, and exports ONNX. Unlike Colab there's no Drive step — everything lands in `/kaggle/working/`, which you download straight from the **Output** pane (top-right "Data" tab, or the file browser on the left) once it's done.

A full run (10k samples, 30 epochs) is roughly 2 hours based on real T4 timings from a prior run — comfortably inside Kaggle's per-session limit. To do a quick end-to-end test first, lower `SAMPLES` and `EPOCHS` in the config cell.

## 1. Check the GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n⚠️  No GPU. Open Settings (right panel) → Accelerator → GPU T4 x2, then Run All again.')

## 2. Config — tweak these, then Run All

In [ ]:
BRANCH  = 'main-uiyymm'   # branch to train from
SAMPLES = 10000           # training plans to generate (try 400 for a quick test)
VAL     = 500             # validation plans
EPOCHS  = 30              # training epochs (try 3 for a quick test)
BATCH   = 8               # lower to 4 if you hit out-of-memory
SIZE    = 512             # training crop size
BASE    = 32              # U-Net width (model capacity)

WORK = '/kaggle/working'  # the only directory Kaggle persists as downloadable output

## 3. Install Node.js (for the generator) + clone the repo

Kaggle's base image doesn't ship Node.js (Colab's does) — installed explicitly here, pinned to 20.x to match what this pipeline is tested against.

In [ ]:
import subprocess

def sh(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

has_node = subprocess.run('command -v node', shell=True, capture_output=True, text=True).returncode == 0
if not has_node:
    print('Installing Node.js 20.x...')
    sh('curl -fsSL https://deb.nodesource.com/setup_20.x | bash -')
    sh('apt-get install -y -qq nodejs')
sh('node --version')

In [ ]:
import os
os.chdir(WORK)
if not os.path.isdir('stride'):
    sh(f'git clone --branch {BRANCH} https://github.com/tiienn/stride.git')
os.chdir(f'{WORK}/stride')
# only the generator's dep is needed (resvg); skip the app's heavy 3D deps
sh('npm install @resvg/resvg-js --no-save --no-audit --no-fund')

## 4. Generate the synthetic dataset

Images + pixel‑perfect masks + ground‑truth JSON. ~110 ms/sample. Runs as a driver that restarts itself in a fresh subprocess every 400 samples (a native memory leak in the SVG renderer otherwise grows unbounded across a big run) and verifies the exact file count before continuing — a partial dataset halts here with a clear error instead of silently training on it.

In [ ]:
import glob

def run(cmd):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)  # raises on non-zero exit -> halts Run All

run(['node', 'ml/generate.mjs', '--count', str(SAMPLES), '--out', 'ml/data/train', '--seed', '1'])
run(['node', 'ml/generate.mjs', '--count', str(VAL), '--out', 'ml/data/val', '--seed', '999'])

n_train = len(glob.glob('ml/data/train/img_*.png'))
n_val = len(glob.glob('ml/data/val/img_*.png'))
print('train images:', n_train)
print('val images:  ', n_val)
assert n_train == SAMPLES, f'train set incomplete: {n_train}/{SAMPLES} images - do not proceed'
assert n_val == VAL, f'val set incomplete: {n_val}/{VAL} images - do not proceed'

## 5. Preview a sample (sanity check)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/train/img_00000.png')); ax[0].set_title('drawing'); ax[0].axis('off')
ax[1].imshow(Image.open('ml/data/train/msk_00000.png')); ax[1].set_title('mask: red=wall green=door blue=window'); ax[1].axis('off')
plt.show()

## 6. Train

Watch the per‑class `val IoU`. Walls climb first; doors/windows lag (rarer pixels) but the class weights compensate. Good v1 targets: wall ≥ 0.85, door/window ≥ 0.6. `best.pt` is saved whenever val mIoU improves.

In [ ]:
os.chdir(f'{WORK}/stride/ml/train')
sh(f'python train.py --data ../data/train --out ../checkpoints '
   f'--epochs {EPOCHS} --batch {BATCH} --size {SIZE} --base {BASE} --val-frac 0.05')
os.chdir(f'{WORK}/stride')

## 7. Try it on a held‑out plan

In [ ]:
os.chdir(f'{WORK}/stride/ml/train')
sh(f'python infer.py --checkpoint ../checkpoints/best.pt --base {BASE} '
   f'--image ../data/val/img_00003.png --out-prefix {WORK}/pred')
os.chdir(f'{WORK}/stride')
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/val/img_00003.png')); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(Image.open(f'{WORK}/pred_mask.png')); ax[1].set_title('model prediction'); ax[1].axis('off')
plt.show()

## 8. Export ONNX (for in‑browser inference in Stride)

In [ ]:
sh('pip install --quiet onnx')
import sys
sys.path.insert(0, f'{WORK}/stride/ml/train')  # explicit, not cwd-dependent import
os.chdir(f'{WORK}/stride/ml/train')
import torch
from model import UNet
m = UNet(4, base=BASE)
m.load_state_dict(torch.load('../checkpoints/best.pt', map_location='cpu'))
m.eval()
torch.onnx.export(m, torch.zeros(1, 3, 512, 512), f'{WORK}/stride-planseg.onnx',
                  input_names=['image'], output_names=['logits'], opset_version=17,
                  dynamo=False, dynamic_axes={'image': {2: 'h', 3: 'w'}, 'logits': {2: 'h', 3: 'w'}})
os.chdir(f'{WORK}/stride')
print('ONNX size: %.1f MB' % (os.path.getsize(f'{WORK}/stride-planseg.onnx') / 1e6))

## 9. Collect the trained model for download

No Drive step needed on Kaggle: anything under `/kaggle/working/` is automatically downloadable. This just gathers both files into one clearly-named folder — after this cell finishes, open the **file browser** (left sidebar, or the "Output" tab on the right for a finished run) and download `stride-model/best.pt` and `stride-model/stride-planseg.onnx` (⋮ menu → Download on each, or download the whole folder as a zip).

In [ ]:
import shutil
out_dir = f'{WORK}/stride-model'
os.makedirs(out_dir, exist_ok=True)
shutil.copy(f'{WORK}/stride/ml/checkpoints/best.pt', out_dir)
shutil.copy(f'{WORK}/stride-planseg.onnx', out_dir)
print(f'Ready to download from: {out_dir}')
sh(f'ls -la {out_dir}')